# Cleaning this A/B Test Dataset

**Dataset:** [Marketing A/B Testing](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing)

Each row of data represents a unique user with complementary meta data regarding the users and their interaction with the A/B test. 

**The initial cleaning is to prepare the exported file to answer the following:**
1. Did the ad group convert at higher rate than the PSA group?
2. Is that difference significant given the sample size?
3. How does conversion rates change with ad frequency?
4. Are certain days of the week better than other at converting users?
5. How many conversions can be attributed to ads instead of the PSAs?

**The Approach:**
1. Remove redundant columns 
2. Rename columns for personal preference and consistency
3. Drop duplicate rows
4. Handle missing/placeholder values
5. Clean individual columns if necessary
6. Export cleaned dataset

## Load Data & First Look

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("../data/marketing_AB.csv")
df

,Unnamed: 0,user id,test group,converted,total ads,most ads day,most ads hour
0,0,1069124,ad,False,130,Monday,20
1,1,1119715,ad,False,93,Tuesday,22
2,2,1144181,ad,False,21,Tuesday,18
3,3,1435133,ad,False,355,Tuesday,10
4,4,1015700,ad,False,276,Friday,14
...,...,...,...,...,...,...,...
588096,588096,1278437,ad,False,1,Tuesday,23
588097,588097,1327975,ad,False,1,Tuesday,23
588098,588098,1038442,ad,False,3,Tuesday,23
588099,588099,1496395,ad,False,1,Tuesday,23


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 588101 entries, 0 to 588100
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   Unnamed: 0     588101 non-null  int64
 1   user id        588101 non-null  int64
 2   test group     588101 non-null  str  
 3   converted      588101 non-null  bool 
 4   total ads      588101 non-null  int64
 5   most ads day   588101 non-null  str  
 6   most ads hour  588101 non-null  int64
dtypes: bool(1), int64(4), str(2)
memory usage: 32.6 MB


All the columns seem to be self explanatory. Right off the back I do want to remove the index column as I already have a unique user id column, so the index isn't necessary here. Also, the column names will be adjusted snake case (my personal preference). Besides that, from first glance everything else seems fine for now.

## Step 1: Drop Unnecessary Index Column

I don't need the index that was a part of the inital CSV file, especially since the `user id` field will be my unique identifier later. 

In [7]:
df = df.drop(columns = ["Unnamed: 0"])
df

,user id,test group,converted,total ads,most ads day,most ads hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14
...,...,...,...,...,...,...
588096,1278437,ad,False,1,Tuesday,23
588097,1327975,ad,False,1,Tuesday,23
588098,1038442,ad,False,3,Tuesday,23
588099,1496395,ad,False,1,Tuesday,23


## Step 2: Restructure Column Names

I prefer snake case for column names.

In [8]:
df.columns = df.columns.str.replace(" ", "_")
df.head()

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14


## Step 3: Drop Duplicate Rows

In [9]:
df.duplicated().value_counts()

False    588101
Name: count, dtype: int64

Good news, no duplicate rows. But let me check to ensure that their is only one row per `user_id`.

In [10]:
df['user_id'].duplicated().value_counts()

user_id
False    588101
Name: count, dtype: int64

Great, only one user per row, exactly what I want. 

## Step 4: Handle Missing Values

In [11]:
df.isna().sum()

user_id          0
test_group       0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
dtype: int64

From the results it looks like we have no missing values, but there could potentially be placeholder values for nulls. With that in mind, I want to get a look at the most frequent values for each column to see if any odd outliers are present.

In [16]:
for col in df.columns:
    print(f"--- Top values in {col} ---")
    print(df[col].value_counts(dropna = False).head(10))
    print("\n")

--- Top values in user_id ---
user_id
1069124    1
1119715    1
1144181    1
1435133    1
1015700    1
1137664    1
1116205    1
1496843    1
1448851    1
1446284    1
Name: count, dtype: int64


--- Top values in test_group ---
test_group
ad     564577
psa     23524
Name: count, dtype: int64


--- Top values in converted ---
converted
False    573258
True      14843
Name: count, dtype: int64


--- Top values in total_ads ---
total_ads
1     56606
2     39827
5     29303
3     28661
4     23426
6     23409
7     19095
15    19031
8     16037
12    15154
Name: count, dtype: int64


--- Top values in most_ads_day ---
most_ads_day
Friday       92608
Monday       87073
Sunday       85391
Thursday     82982
Saturday     81660
Wednesday    80908
Tuesday      77479
Name: count, dtype: int64


--- Top values in most_ads_hour ---
most_ads_hour
13    47655
12    47298
11    46210
14    45648
15    44683
10    38939
16    37567
17    34988
18    32323
9     31004
Name: count, dtype: int64




Everything looks good from the previous results, but for the numerical columns `total_ads` and `most_ads_hour`, I am going to dig a little deeper looking at the numbers to ensure there are no crazy outliers.

In [18]:
print(df["total_ads"].describe())
print(df["most_ads_hour"].describe())

count    588101.000000
mean         24.820876
std          43.715181
min           1.000000
25%           4.000000
50%          13.000000
75%          27.000000
max        2065.000000
Name: total_ads, dtype: float64
count    588101.000000
mean         14.469061
std           4.834634
min           0.000000
25%          11.000000
50%          14.000000
75%          18.000000
max          23.000000
Name: most_ads_hour, dtype: float64


Everything looks good regarding any placeholder values in the dataset.

## Step 5: Clean Individual Columns

Fortunately, it looks like everything is cleaned and ready to export as I want it. So nothing else to do here. 

## Export

In [14]:
df.to_csv("cleaned_marketing_AB.csv", index = False)

In [15]:
pd.read_csv("cleaned_marketing_AB.csv")

,user_id,test_group,converted,total_ads,most_ads_day,most_ads_hour
0,1069124,ad,False,130,Monday,20
1,1119715,ad,False,93,Tuesday,22
2,1144181,ad,False,21,Tuesday,18
3,1435133,ad,False,355,Tuesday,10
4,1015700,ad,False,276,Friday,14
...,...,...,...,...,...,...
588096,1278437,ad,False,1,Tuesday,23
588097,1327975,ad,False,1,Tuesday,23
588098,1038442,ad,False,3,Tuesday,23
588099,1496395,ad,False,1,Tuesday,23
